# ICU Extubation Decision-Making Model Training & Testing

In [22]:
import sys
import os
# Add parent directory to path (project root)
sys.path.append(os.path.abspath('..'))

In [ ]:
from ConMedRL.conmedrl import *
from ConMedRL.data_loader import *

In [3]:
em_configuration = RLConfigurator()

In [ ]:
em_configuration.choose_config_method()

In [ ]:
em_configuration.config.memory_capacity

In [ ]:
outcome_table_train = pd.read_csv('extubation_sample_outcome_table_train.csv')
state_var_table = pd.read_csv('extubation_sample_state_var_table_train.csv')

outcome_table_val = pd.read_csv('extubation_sample_outcome_table_val.csv')
outcome_table_val_select = pd.read_csv('extubation_sample_outcome_table_val_select.csv')

state_var_table_val = pd.read_csv('extubation_sample_state_var_table_val.csv')
state_var_table_val_select = pd.read_csv('extubation_sample_state_var_table_val_select.csv')

# New build_dataset workflows expose these values on `bundle`. The fallback
# preserves compatibility with the published CSV-only example.
bundle = globals().get('bundle')
state_dim = bundle.state_dim if bundle is not None else state_var_table.shape[1]
action_name = bundle.loader_action if bundle is not None else 'extubation_action'
action_dim = bundle.action_dim if bundle is not None else 2
num_constraints = bundle.num_constraints if bundle is not None else 1

In [7]:
terminal_state = np.zeros(state_var_table.shape[1])

In [ ]:
len(terminal_state)

In [ ]:
outcome_table_train.columns

In [ ]:
state_var_table.columns

In [11]:
train_data_loader = TrainDataLoader(cfg = em_configuration.config, 
                                    outcome_table = outcome_table_train, 
                                    state_var_table = state_var_table, 
                                    terminal_state = terminal_state)

In [ ]:
train_data_loader.data_buffer_train(action_name = action_name,
                                    done_condition = None,
                                    num_constraint = num_constraints)

In [14]:
val_data_loader = ValTestDataLoader(cfg = em_configuration.config, 
                                    outcome_table_select = outcome_table_val_select, 
                                    state_var_table_select = state_var_table_val_select, 
                                    outcome_table = outcome_table_val, 
                                    state_var_table = state_var_table_val, 
                                    terminal_state = terminal_state)

In [ ]:
val_data_loader.data_buffer(action_name = action_name,
                            done_condition = None,
                            num_constraint = num_constraints)

In [16]:
ocrl_training = RLTraining(cfg = em_configuration.config,
                           state_dim = state_dim,
                           action_dim = action_dim,
                           train_data_loader = train_data_loader.data_torch_loader_train,
                           val_data_loader = val_data_loader.data_torch_loader)

In [ ]:
# Building the FQI agent
fqi_agent = ocrl_training.fqi_agent_config(hidden_layers = [128, 128], 
                                           weight_decay = None, 
                                           seed = 1) 

# Building the FQE agents
fqe_agent_obj = ocrl_training.fqe_agent_config(eval_agent = fqi_agent, 
                                                 hidden_layers = [1000], 
                                                 weight_decay = None, 
                                                 eval_target = 'obj', 
                                                 seed = 1) 

fqe_agent_con_0 = ocrl_training.fqe_agent_config(eval_agent = fqi_agent, 
                                                 hidden_layers = [1000], 
                                                 weight_decay = None, 
                                                 eval_target = 0, 
                                                 seed = 1) 

In [ ]:
ocrl_training.train(agent_fqi = fqi_agent, 
                    agent_fqe_obj = fqe_agent_obj, 
                    agent_fqe_con_list = [fqe_agent_con_0], 
                    constraint = True,
                    save_num = 100,
                    z_value = 1.96)